# Lean 4 Jupyter Kernel Demo (`leanprover-community/repl`)

이 노트북은 [`leanprover-community/repl`](https://github.com/leanprover-community/repl)을 백엔드로 사용하는 **Lean 4 Jupyter 커널**의 주요 기능을 시연합니다.

- **Command Mode**: 선언(`def`, `structure`, `theorem`), 계산(`#eval`), 타입/공리 검증(`#check`, `#print axioms`)
- **Smart Branching**: 셀 재정의 시 자동 스냅샷 분기(Auto-Backtrack) 및 에러 발생 시 환경 자동 보존
- **Interactive Tactic Mode**: `%%tactic` / `%tactic` 매직을 통한 단계별 목표(`⊢`) 탐색 및 증명 검증
- **File Mode & Snapshot History**: `%load`, `%env`, `%history`, `%pickle` / `%unpickle`

## 1. 기본 선언 및 계산 (Command Mode)

In [1]:
def fib : Nat → Nat
  | 0 => 0
  | 1 => 1
  | n + 2 => fib n + fib (n + 1)

#check fib
#eval (List.range 10).map fib

fib : Nat → Nat

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

## 2. 형식 증명 및 공리 검증 (Formal Proof & Axiom Check)

In [2]:
theorem fib_pos_succ (n : Nat) : 1 ≤ fib (n + 1) := by
  induction n with
  | zero => decide
  | succ k ih =>
    simp [fib]
    omega

#print axioms fib_pos_succ

'fib_pos_succ' depends on axioms: [propext, Quot.sound]

## 3. 대화형 전술 모드 (Interactive Tactic Mode)

`by sorry`로 열린 목표(`proofState`)를 생성한 뒤, `%%tactic` / `%tactic`으로 단계별 증명을 수행하고 최종 정리를 완성합니다.

In [3]:
theorem currying_equiv (P Q R : Prop) : (P ∧ Q → R) ↔ (P → Q → R) := by
  sorry

[open goal · proofState #0 · line 2:2]
P Q R : Prop
⊢ P ∧ Q → R ↔ P → Q → R

In [4]:
%%tactic
constructor
· intro h hp hq
  exact h ⟨hp, hq⟩

[proofState #1 · 1 goal (Incomplete: open goals remain)]
case mpr
P Q R : Prop
⊢ (P → Q → R) → P ∧ Q → R

In [5]:
%%tactic
intro h hpq
exact h hpq.1 hpq.2

✓ Goals accomplished! (proofState #2, Completed)

위 전술 탐색이 `Completed`로 검증되었으므로, 동일한 이름으로 완성된 정리(무-`sorry`)를 재정의합니다. 커널의 **자동 스냅샷 분기(Auto-Backtrack)** 기능이 이전 `sorry` 선언 이전 스냅샷에서 깔끔하게 분기합니다.

In [6]:
theorem currying_equiv (P Q R : Prop) : (P ∧ Q → R) ↔ (P → Q → R) := by
  constructor
  · intro h hp hq
    exact h ⟨hp, hq⟩
  · intro h hpq
    exact h hpq.1 hpq.2

#print axioms currying_equiv

ℹ Redefined `currying_equiv` by branching from env #1 → env #4.

'currying_equiv' does not depend on any axioms

## 4. 외부 `.lean` 파일 로드 (`%load`) 및 세션 상태 조회 (`%env`, `%history`)

In [7]:
%load ../examples/Arithmetic.lean

ℹ Loaded `/Users/yiinsu/Library/CloudStorage/GoogleDrive-ii3289857@gmail.com/My Drive/Github/proof-notebook/examples/Arithmetic.lean` → env #5.

'gaussSum_eq' depends on axioms: [propext]

In [8]:
#eval gaussSum 100

5050

In [9]:
%history

[Snapshot History]
Environment Snapshots:
    env #0 (from root): def fib : Nat → Nat
    env #1 (from #0): theorem fib_pos_succ (n : Nat) : 1 ≤ fib (n + 1) := by
    env #2 (from #1): theorem currying_equiv (P Q R : Prop) : (P ∧ Q → R) ↔ (P → Q ...
    env #4 (from #1): theorem currying_equiv (P Q R : Prop) : (P ∧ Q → R) ↔ (P → Q ...
    env #5 (from #4): file: /Users/yiinsu/Library/CloudStorage/GoogleDrive-ii3289857@gmail.com/My Drive/Github/proof-notebook/examples/Arithmetic.lean
  * env #6 (from #5): #eval gaussSum 100

Proof State Snapshots:
    proofState #0 (from env #2) [1 goal(s), Incomplete: open goals remain]: (sorry)
    proofState #1 (from #0) [1 goal(s), Incomplete: open goals remain]: constructor ...
  * proofState #2 (from #1) [0 goal(s), Completed]: intro h hpq ...